# 面试问题：怎样实现可靠的 JSON Structured Output，约束解码与事后修复有什么区别？

**一句话回答**：先把业务 schema 编译成语法状态机；每一步根据当前状态屏蔽不能继续形成合法输出的 token，只在允许集合中采样，结束后再做类型、范围和跨字段语义校验。约束解码保证语法，不保证事实与业务正确；事后 repair 可能悄悄改变含义，只适合低风险回退并必须再次验证。

本 Notebook 对有限 JSON schema 枚举合法串、构建 trie、实现逐字符 logits mask 与采样，再覆盖语义约束、转义、资源上限和版本合同。

In [ ]:
import hashlib, json, math
import numpy as np

SEED109=10901; rng109=np.random.default_rng(SEED109)
LABELS109=("positive","negative","neutral"); SCORES109=tuple(range(10)); EOS109="<eos>"
assert len(LABELS109)*len(SCORES109)==30
assert json.loads('{"label":"positive","score":9}')["score"]==9
assert SEED109==10901

## 1. Schema 同时包含语法和业务合同

JSON parser 只保证语法；schema 还需限定 required、禁止额外字段、枚举、整数范围与最大长度。类型不做隐式转换：`"9"` 不是整数 9。最终 validator 是安全边界，即使解码器声称已约束也必须重验。

In [ ]:
def validate109(text):
    try: obj=json.loads(text)
    except (json.JSONDecodeError,TypeError): return False,"invalid_json"
    if not isinstance(obj,dict) or set(obj)!={"label","score"}: return False,"object_shape"
    if obj["label"] not in LABELS109: return False,"label_enum"
    if isinstance(obj["score"],bool) or not isinstance(obj["score"],int) or not 0<=obj["score"]<=9: return False,"score_range"
    return True,obj
assert validate109('{"label":"positive","score":9}')[0]
assert validate109('{"label":"other","score":9}')==(False,"label_enum")
assert validate109('{"label":"positive","score":"9"}')==(False,"score_range")

## 2. 有限 schema 可以编译成合法语言

为了把机制讲清楚，这里枚举 30 个合法紧凑 JSON。真实 JSON Schema 可能包含递归 object/array，需要 DFA、pushdown/CFG 或增量 parser。序列化选项、字段顺序和 Unicode 策略必须固定，否则训练示例与解码语言不一致。

In [ ]:
candidates109=[json.dumps({"label":label,"score":score},ensure_ascii=False,separators=(",",":")) for label in LABELS109 for score in SCORES109]
assert len(candidates109)==30 and len(set(candidates109))==30
assert all(validate109(x)[0] for x in candidates109)
assert candidates109[0]=='{"label":"positive","score":0}'

## 3. Trie 就是有限语言的解码状态机

每个节点代表一个合法前缀，边是下一个字符，终结节点允许 EOS。若当前 prefix 不在 trie 中，说明此前已走入死路。生产 tokenizer 的 token 可能包含多个字符/字节，需要模拟 token 拼接后是否仍是合法前缀，而不能只看首字符。

In [ ]:
END109="__end__"; trie109={}
for text in candidates109:
    node=trie109
    for ch in text: node=node.setdefault(ch,{})
    node[END109]=True
def trie_node109(prefix):
    node=trie109
    for ch in prefix:
        if ch not in node: return None
        node=node[ch]
    return node
def allowed_next109(prefix):
    node=trie_node109(prefix)
    if node is None: return set()
    out={k for k in node if k!=END109}
    if END109 in node: out.add(EOS109)
    return out
assert allowed_next109("")== {"{"}
assert allowed_next109(candidates109[0])=={EOS109}
assert allowed_next109("!")==set()

## 4. Logits mask 在采样前应用

把不允许 token 的 logit 设为负无穷，再在剩余 token 上 softmax/top-p。若 allowed set 为空，不应随便解除约束，应记录 grammar error 并重试或失败。下面字符词表故意加入模型最偏爱的非法 `!`。

In [ ]:
VOCAB109=sorted(set("".join(candidates109))|{"!"})+[EOS109]; tok_to_id109={t:i for i,t in enumerate(VOCAB109)}
def mask_logits109(prefix,logits):
    allowed=allowed_next109(prefix); out=np.full_like(np.asarray(logits,float),-np.inf)
    for tok in allowed: out[tok_to_id109[tok]]=logits[tok_to_id109[tok]]
    return out
logits_probe109=np.zeros(len(VOCAB109)); logits_probe109[tok_to_id109["!"]]=100; masked_probe109=mask_logits109("",logits_probe109)
assert np.argmax(logits_probe109)==tok_to_id109["!"]
assert np.argmax(masked_probe109)==tok_to_id109["{"]
assert np.isneginf(masked_probe109[tok_to_id109["!"]])

## 5. 受约束 Greedy 的端到端实现

假模型在每步都最喜欢非法 token，其次喜欢目标串的下一个字符。无约束首步立即输出 `!`；约束后只能沿 trie 行走并以 EOS 结束。这验证的是结构保证，不代表模型语义质量。

In [ ]:
target109=json.dumps({"label":"positive","score":9},separators=(",",":")); prefix109=""; generated109=""
for step in range(200):
    logits=np.zeros(len(VOCAB109)); logits[tok_to_id109["!"]]=20
    desired=EOS109 if len(prefix109)==len(target109) else target109[len(prefix109)]; logits[tok_to_id109[desired]]=10
    masked=mask_logits109(prefix109,logits); chosen=VOCAB109[int(np.argmax(masked))]
    if chosen==EOS109: break
    generated109+=chosen; prefix109=generated109
assert generated109==target109
assert validate109(generated109)[0]
assert step<100 and chosen==EOS109

## 6. 采样也必须在条件分布上重新归一化

mask 后只对有限 logits 做稳定 softmax。temperature/top-p 都在 allowed token 集合内工作；不同随机 seed 可得到不同合法对象。若某一步只有一个允许 token，采样自然退化为确定性选择。

In [ ]:
def sample_json109(seed):
    rg=np.random.default_rng(seed); prefix=""
    for _ in range(200):
        allowed=sorted(allowed_next109(prefix)); scores=rg.normal(size=len(allowed)); m=scores.max(); probs=np.exp(scores-m); probs/=probs.sum(); tok=allowed[int(rg.choice(len(allowed),p=probs))]
        if tok==EOS109: return prefix
        prefix+=tok
    raise RuntimeError("decode_budget")
samples109=[sample_json109(i) for i in range(20)]
assert all(validate109(s)[0] for s in samples109)
assert sample_json109(7)==sample_json109(7)
assert len(set(samples109))>1

## 7. 语法合法不等于业务语义合法

例如规定 positive 的 score 至少 5、negative 至多 4。可以把有限约束编译进 grammar，或在生成后 validator 拒绝；涉及数据库状态、权限或事实的约束只能在宿主校验。repair 不得擅自把 3 改成 5 后当成原意。

In [ ]:
def semantic109(text):
    ok,obj=validate109(text)
    if not ok: return False,obj
    if obj["label"]=="positive" and obj["score"]<5: return False,"positive_score"
    if obj["label"]=="negative" and obj["score"]>4: return False,"negative_score"
    return True,obj
assert semantic109('{"label":"positive","score":9}')[0]
assert semantic109('{"label":"positive","score":2}')==(False,"positive_score")
assert validate109('{"label":"negative","score":8}')[0] and not semantic109('{"label":"negative","score":8}')[0]

## 8. 转义、资源限制、fallback 与版本

通用实现还要处理字符串转义、UTF-8、嵌套深度、数组长度、最大输出 token 和 tokenizer 特殊 token。严格约束失败时可有限重试、切换更小 schema 或拒绝；事后 repair 的原文与修复结果都要审计。schema、编译器、tokenizer 和生成参数共同组成版本。

In [ ]:
escaped109=json.dumps({"text":"他说：\"你好\""},ensure_ascii=False); roundtrip109=json.loads(escaped109)
manifest109={"schema":1,"json_schema":"sentiment-v3","compiler":"finite-trie-v1","tokenizer":"char-demo","max_chars":128,"on_empty_mask":"fail","repair":"disabled"}; digest109=hashlib.sha256(json.dumps(manifest109,sort_keys=True).encode()).hexdigest()
assert roundtrip109["text"]=='他说："你好"'
assert len(max(candidates109,key=len))<manifest109["max_chars"]
assert len(digest109)==64 and manifest109["on_empty_mask"]=="fail"

## 面试总结

回答主线是：**schema → 增量语法状态 → tokenizer/token 合法性 → mask 后重归一化 → EOS/空集合 → 最终类型与语义验证 → 有限 retry/fail → 全链路版本**。约束解码解决“能否解析”，事实、权限和业务正确性仍必须由外部证据与确定性代码保证。

延伸阅读：[JSON RFC 8259](https://www.rfc-editor.org/rfc/rfc8259)、[JSON Schema 规范](https://json-schema.org/specification)、[Grammar-Constrained Decoding](https://arxiv.org/abs/2305.13971)。